In [2]:
import polars as pl
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# show more columns/rows in Polars printouts
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_str_lengths(200)

df_full = pl.read_csv("player_rankings_pca.csv")
feature_cols = [
    "net_dpr", "nade_ratio", "headshotperhit", "engagement_rate",
    "survivability", "efficiency",
    "clutch_rate", "win_rate", "match_win_rate"
]

# keep player separately
players = df_full.get_column("player")

# features only
df = df_full.select(feature_cols)

# use numpy array for sklearn
X = df.to_numpy()

results = []
cluster_size_data = {}
labels_by_k = {}

for k in range(2, 6):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)

    labels_by_k[k] = labels

    unique, counts = np.unique(labels, return_counts=True)
    cluster_counts = dict(zip(unique.tolist(), counts.tolist()))
    min_cluster_size = counts.min()
    n_singletons = np.sum(counts == 1)
    smallest_pct = min_cluster_size / len(labels)

    sil = silhouette_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    db = davies_bouldin_score(X, labels)

    print(f"\n=== k = {k} ===")
    print("Cluster sizes:", cluster_counts)
    print(f"Smallest cluster size: {min_cluster_size}")
    print(f"Singleton clusters: {n_singletons}")
    print(f"Silhouette: {sil:.4f}, CH: {ch:.2f}, DB: {db:.4f}")

    if n_singletons > 0:
        print("Warning: clustering still contains singleton outlier clusters.")
    elif smallest_pct < 0.02:
        print("Warning: clustering is very imbalanced.")

    results.append({
        "n_clusters": k,
        "silhouette_score": sil,
        "calinski_harabasz": ch,
        "davies_bouldin": db,
        "min_cluster_size": int(min_cluster_size),
        "singleton_clusters": int(n_singletons)
    })

    cluster_size_data[k] = cluster_counts

results_df = pd.DataFrame(results)
print("\nSummary table:")
print(results_df)

chosen_k = 4
labels = labels_by_k[chosen_k]

clustered_df = pl.DataFrame({
    "player": players,
    "cluster": labels,
    **{col: df.get_column(col) for col in df.columns}
})

print(clustered_df)

player_clusters = clustered_df.select(["player", "cluster"]).sort(["cluster", "player"])
print(player_clusters)

cluster_means = (
    clustered_df
    .group_by("cluster")
    .agg(
        pl.exclude(["player", "cluster"]).mean()
    )
    .sort("cluster")
)

print("\nCluster means:")
print(cluster_means)

cluster_summary = (
    clustered_df
    .group_by("cluster")
    .agg(
        [pl.len().alias("n_players")] +
        [pl.col(c).mean().alias(f"{c}_mean") for c in feature_cols] +
        [pl.col(c).std().alias(f"{c}_std") for c in feature_cols] +
        [pl.col(c).min().alias(f"{c}_min") for c in feature_cols] +
        [pl.col(c).max().alias(f"{c}_max") for c in feature_cols]
    )
    .sort("cluster")
)

print("\nCluster summary:")
print(cluster_summary)


=== k = 2 ===
Cluster sizes: {0: 5864, 1: 5845}
Smallest cluster size: 5845
Singleton clusters: 0
Silhouette: 0.4905, CH: 16208.31, DB: 0.7019

=== k = 3 ===
Cluster sizes: {0: 5815, 1: 2991, 2: 2903}
Smallest cluster size: 2903
Singleton clusters: 0
Silhouette: 0.4618, CH: 18253.68, DB: 0.6788

=== k = 4 ===
Cluster sizes: {0: 4178, 1: 1842, 2: 1616, 3: 4073}
Smallest cluster size: 1616
Singleton clusters: 0
Silhouette: 0.4282, CH: 19053.86, DB: 0.7011

=== k = 5 ===
Cluster sizes: {0: 2792, 1: 2899, 2: 1036, 3: 1064, 4: 3918}
Smallest cluster size: 1036
Singleton clusters: 0
Silhouette: 0.4079, CH: 19554.96, DB: 0.7224

Summary table:
   n_clusters  silhouette_score  calinski_harabasz  davies_bouldin  \
0           2          0.490482       16208.305326        0.701871   
1           3          0.461833       18253.683064        0.678818   
2           4          0.428191       19053.863740        0.701127   
3           5          0.407905       19554.959433        0.722359   

   

In [3]:
import polars as pl
import pandas as pd
import numpy as np

from sklearn.cluster import SpectralClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# show more columns/rows in Polars printouts
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_str_lengths(200)

df_full = pl.read_csv("player_rankings_pca.csv")

feature_cols = [
    "net_dpr", "nade_ratio", "headshotperhit", "engagement_rate",
    "survivability", "efficiency",
    "clutch_rate", "win_rate", "match_win_rate"
]

# keep player separately
players = df_full.get_column("player")

# features only
df = df_full.select(feature_cols)

# use numpy array for sklearn
X = df.to_numpy()

results = []
cluster_size_data = {}
labels_by_k = {}

for k in range(2, 6):
    model = SpectralClustering(
        n_clusters=k,
        affinity="nearest_neighbors",
        n_neighbors=10,
        assign_labels="kmeans",
        random_state=42
    )
    labels = model.fit_predict(X)

    labels_by_k[k] = labels

    unique, counts = np.unique(labels, return_counts=True)
    cluster_counts = dict(zip(unique.tolist(), counts.tolist()))
    min_cluster_size = counts.min()
    n_singletons = np.sum(counts == 1)
    smallest_pct = min_cluster_size / len(labels)

    sil = silhouette_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    db = davies_bouldin_score(X, labels)

    print(f"\n=== k = {k} ===")
    print("Cluster sizes:", cluster_counts)
    print(f"Smallest cluster size: {min_cluster_size}")
    print(f"Singleton clusters: {n_singletons}")
    print(f"Silhouette: {sil:.4f}, CH: {ch:.2f}, DB: {db:.4f}")

    if n_singletons > 0:
        print("Warning: clustering still contains singleton outlier clusters.")
    elif smallest_pct < 0.02:
        print("Warning: clustering is very imbalanced.")

    results.append({
        "n_clusters": k,
        "silhouette_score": sil,
        "calinski_harabasz": ch,
        "davies_bouldin": db,
        "min_cluster_size": int(min_cluster_size),
        "singleton_clusters": int(n_singletons)
    })

    cluster_size_data[k] = cluster_counts

results_df = pd.DataFrame(results)
print("\nSummary table:")
print(results_df)

chosen_k = 4
labels = labels_by_k[chosen_k]

clustered_df = pl.DataFrame({
    "player": players,
    "cluster": labels,
    **{col: df.get_column(col) for col in df.columns}
})

print(clustered_df)

player_clusters = clustered_df.select(["player", "cluster"]).sort(["cluster", "player"])
print(player_clusters)

cluster_means = (
    clustered_df
    .group_by("cluster")
    .agg(
        pl.exclude(["player", "cluster"]).mean()
    )
    .sort("cluster")
)

print("\nCluster means:")
print(cluster_means)

cluster_summary = (
    clustered_df
    .group_by("cluster")
    .agg(
        [pl.len().alias("n_players")] +
        [pl.col(c).mean().alias(f"{c}_mean") for c in feature_cols] +
        [pl.col(c).std().alias(f"{c}_std") for c in feature_cols] +
        [pl.col(c).min().alias(f"{c}_min") for c in feature_cols] +
        [pl.col(c).max().alias(f"{c}_max") for c in feature_cols]
    )
    .sort("cluster")
)

print("\nCluster summary:")
print(cluster_summary)


=== k = 2 ===
Cluster sizes: {0: 5269, 1: 6440}
Smallest cluster size: 5269
Singleton clusters: 0
Silhouette: 0.4891, CH: 16024.09, DB: 0.6997

=== k = 3 ===
Cluster sizes: {0: 6504, 1: 2913, 2: 2292}
Smallest cluster size: 2292
Singleton clusters: 0
Silhouette: 0.4616, CH: 17721.62, DB: 0.6618

=== k = 4 ===
Cluster sizes: {0: 4600, 1: 4059, 2: 1393, 3: 1657}
Smallest cluster size: 1393
Singleton clusters: 0
Silhouette: 0.4304, CH: 18780.58, DB: 0.6858

=== k = 5 ===
Cluster sizes: {0: 3884, 1: 3225, 2: 852, 3: 1058, 4: 2690}
Smallest cluster size: 852
Singleton clusters: 0
Silhouette: 0.4111, CH: 19433.17, DB: 0.7128

Summary table:
   n_clusters  silhouette_score  calinski_harabasz  davies_bouldin  \
0           2          0.489105       16024.089030        0.699708   
1           3          0.461606       17721.621199        0.661799   
2           4          0.430366       18780.579243        0.685801   
3           5          0.411053       19433.165112        0.712776   

   mi